# 交叉熵、KL 散度、Entropy balancing 与加权估计

## 1. 从“分布有多不确定”到“模型错得有多贵”

前两篇讨论了 entropy 和 conditional entropy。它们关心的是概率分布本身的不确定性，以及已知某些信息后不确定性下降多少。现在换一个角度：如果真实世界有一个分布 $P$，模型给出了另一个分布 $Q$，我们如何衡量模型分布 $Q$ 描述真实分布 $P$ 的代价？

这就是 cross-entropy 和 KL divergence 的位置。它们不是在问“真实世界本身有多不确定”，而是在问“用模型给出的概率去解释真实结果时，平均要付出多少额外成本”。这也是为什么分类模型、神经网络、语言模型和概率预测中经常会出现 cross-entropy loss。


## 2. 二分类中的 cross-entropy loss

考虑一个二分类问题，$y\in\{0,1\}$。模型给出 $P(y=1)=\hat p$。若真实标签是 $y=1$，模型希望 $\hat p$ 越接近 1 越好；若真实标签是 $y=0$，模型希望 $\hat p$ 越接近 0 越好。

二分类 cross-entropy loss 写为：

$$
CE(y,\hat p)=-\left[y\log \hat p+(1-y)\log(1-\hat p)\right]
$$

如果 $y=1$，损失变成 $-\log \hat p$。当模型给真实发生的类别很高概率时，损失很小；当模型给真实类别很低概率时，损失会急剧变大。

![](./figs/entropy_cross_fig01_log_loss.png){width="72%"}


In [ ]:
# 这个代码块可以独立运行，用于计算二分类 cross-entropy。
import numpy as np


def binary_cross_entropy(y, p):
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return -(y * np.log(p) + (1-y) * np.log(1-p))

examples = [(1, 0.9), (1, 0.1), (0, 0.1), (0, 0.9)]
for y, p in examples:
    print(f"y={y}, p_hat={p:.1f}, loss={binary_cross_entropy(y, p):.3f}")


## 3. Cross-entropy 与 MLE 的关系

这部分特别适合和计量经济学中的 MLE 做连接。对于 Bernoulli 模型，若样本独立，log-likelihood 为：

$$
\ell(\theta)=\sum_{i=1}^{n}\left[y_i\log p_i(\theta)+(1-y_i)\log(1-p_i(\theta))\right]
$$

平均 negative log-likelihood 为：

$$
-\frac{1}{n}\ell(\theta)=\frac{1}{n}\sum_{i=1}^{n}CE(y_i,p_i(\theta))
$$

所以，在二分类模型中，最小化 cross-entropy loss 与最大化 Bernoulli log-likelihood 是同一件事的两种表述。机器学习更常说 loss minimization，计量经济学更常说 likelihood maximization。

::: {.callout-warning}
### 注意：likelihood 不是概率的同义词

当数据固定、参数变化时，likelihood 是关于参数的函数。对于离散模型，给定参数下可以把样本 likelihood 写成联合概率；对于连续模型，则是联合密度。密度值可以大于 1，因此不能简单说“似然值就是概率”。更稳妥的说法是：log-likelihood 衡量样本观测值对不同参数取值的相对支持程度。
:::


## 4. 熵、交叉熵与 KL 散度

设真实分布为 $P$，模型预测分布为 $Q$。三者定义如下：

$$
H(P)=-\sum_x P(x)\log_2 P(x)
$$

$$
H(P,Q)=-\sum_x P(x)\log_2 Q(x)
$$

$$
D_{KL}(P||Q)=\sum_x P(x)\log_2\frac{P(x)}{Q(x)}
$$

它们之间有一个非常重要的关系：

$$
H(P,Q)=H(P)+D_{KL}(P||Q)
$$

这意味着，cross-entropy 可以拆成两部分：一部分是现实世界本身的不确定性 $H(P)$，另一部分是模型分布 $Q$ 偏离真实分布 $P$ 所产生的额外代价 $D_{KL}(P||Q)$。

![](./figs/entropy_cross_fig02_ce_kl_relation.png){width="72%"}

::: {.callout-note}
### 一个直观解释

$H(P)$ 是真实世界本身的复杂度，模型无法把它消除；$D_{KL}(P||Q)$ 是模型错配带来的额外成本，是模型可以努力降低的部分。训练分类模型时，真实标签给定后，最小化 cross-entropy 实际上是在推动预测分布 $Q$ 更接近真实分布 $P$。
:::


## 5. 多分类与概率预测

在多分类问题中，真实标签通常写成 one-hot 向量 $y_{ik}$，模型给出每个类别的预测概率 $\hat p_{ik}$。样本 $i$ 的 cross-entropy loss 为：

$$
CE_i=-\sum_{k=1}^{K}y_{ik}\log \hat p_{ik}
$$

如果真实类别是 $k^*$，那么只有 $y_{ik^*}=1$，其他类别为 0，公式就变成：

$$
CE_i=-\log \hat p_{ik^*}
$$

这说明，cross-entropy 并不只看预测类别是否正确，还看模型给真实类别分配了多大概率。两个模型都把样本判为同一类，但一个给真实类别 0.90 的概率，另一个只给 0.51 的概率，cross-entropy 会认为前者更好。这一点对概率预测非常重要。


## 6. Entropy balancing：为什么加权也会用 entropy？

在因果推断中，我们常遇到处理组和控制组不可比的问题。例如，接受某项政策的企业可能规模更大、杠杆更高、成长性更强；如果直接比较处理组和控制组，估计结果可能混入样本选择差异。

IPW 的思路是先估计 propensity score，再用逆概率权重构造一个伪总体。Entropy balancing 的思路不同：它直接寻找一组控制组权重，使加权后的控制组在协变量矩上与处理组平衡，同时让权重尽量不要偏离原始权重。

一个简化写法是：

$$
\min_{w_i}\sum_{i:D_i=0}w_i\log\left(\frac{w_i}{q_i}\right)
$$

subject to

$$
\sum_{i:D_i=0}w_i X_i=\bar X_T,\quad \sum_{i:D_i=0}w_i=1,\quad w_i\geq 0
$$

其中，$q_i$ 是基准权重，$w_i$ 是需要求解的新权重，$\bar X_T$ 是处理组协变量均值。目标函数中的 $w_i\log(w_i/q_i)$ 与 KL divergence 形式相近，含义是让新权重相对原始权重的扭曲尽量小。


## 7. 平衡诊断：不要只看权重生成成功

Entropy balancing 的优势是可以直接让指定协变量矩达到平衡，但这不意味着任何结果都可靠。仍然要检查两个问题。

第一，加权后协变量是否真的平衡。常见做法是报告加权前后的 standardized mean differences (SMD)。经验上，$|SMD|<0.10$ 常被视为较好的平衡状态。

![](./figs/entropy_cross_fig03_balance_before_after.png){width="72%"}

第二，权重是否过度集中。如果少数控制组样本获得极大权重，估计会高度依赖这些样本，方差可能变大，外推风险也会增加。因此，除了平衡图，还应检查权重分布、最大权重、有效样本量 (ESS) 等诊断指标。

![](./figs/entropy_cross_fig04_weight_distribution.png){width="72%"}


## 8. IPW 与 entropy balancing 的关系

IPW 和 entropy balancing 都是加权方法，但它们解决问题的路径不同。

IPW 从 treatment assignment mechanism 出发：先估计 $e(X)=P(D=1|X)$，再构造 $D/e(X)$ 或 $(1-D)/(1-e(X))$ 形式的权重。它的目标是通过逆概率加权恢复一个可比的伪总体。

Entropy balancing 从 covariate balance 出发：不一定先估计 treatment probability，而是直接求一组权重，使控制组加权后的协变量矩与处理组匹配。它的目标是用尽量小的权重扭曲换取明确的平衡约束。

可以这样理解：IPW 更像“先建模，再加权”；entropy balancing 更像“直接规定平衡目标，再反推出权重”。在实证研究中，二者不是互斥的。研究者需要根据识别设计、样本量、协变量维度和权重稳定性来选择方法。


## 9. 小结

这一篇把 entropy 相关概念扩展到预测损失和样本加权。

- Entropy $H(P)$ 衡量真实分布本身的平均信息量。
- Cross-entropy $H(P,Q)$ 衡量用模型分布 $Q$ 描述真实分布 $P$ 的平均代价。
- KL divergence $D_{KL}(P||Q)$ 衡量模型错配带来的额外成本。
- 二分类 cross-entropy 与 Bernoulli 模型的平均 negative log-likelihood 等价。
- Entropy balancing 用 entropy-like criterion 寻找“最小扭曲”的样本权重，以实现协变量平衡。
- IPW 和 entropy balancing 都是重新分配样本权重，但一个从 treatment probability 出发，一个从 covariate balance 出发。

至此，三篇推文形成了一条完整主线：第 1 篇讨论一个分布本身有多不确定，第 2 篇讨论已知信息后不确定性减少多少，第 3 篇讨论模型预测分布与真实分布之间的错配代价，以及这种思想如何进入加权估计。


## 参考文献

- Kullback, S., & Leibler, R. A. (1951). On information and sufficiency. *The Annals of Mathematical Statistics*, 22(1), 79-86. [Link](https://doi.org/10.1214/aoms/1177729694), [PDF](https://projecteuclid.org/journals/annals-of-mathematical-statistics/volume-22/issue-1/On-Information-and-Sufficiency/10.1214/aoms/1177729694.full), [Google](https://scholar.google.com/scholar?q=On+Information+and+Sufficiency+Kullback+Leibler).
- Hainmueller, J. (2012). Entropy balancing for causal effects: A multivariate reweighting method to produce balanced samples in observational studies. *Political Analysis*, 20(1), 25-46. [Link](https://doi.org/10.1093/pan/mpr025), [PDF](https://www.cambridge.org/core/services/aop-cambridge-core/content/view/220E4FC838066552B53128E647E4FAA7/S1047198700012351a.pdf/entropy-balancing-for-causal-effects-a-multivariate-reweighting-method-to-produce-balanced-samples-in-observational-studies.pdf), [Google](https://scholar.google.com/scholar?q=Entropy+Balancing+for+Causal+Effects+A+Multivariate+Reweighting+Method+to+Produce+Balanced+Samples+in+Observational+Studies).
- Hirano, K., Imbens, G. W., & Ridder, G. (2003). Efficient estimation of average treatment effects using the estimated propensity score. *Econometrica*, 71(4), 1161-1189. [Link](https://doi.org/10.1111/1468-0262.00442), [PDF](https://weblaw.usc.edu/centers/class/class-workshops/cleo-working-papers/documents/C02_13_paper.pdf), [Google](https://scholar.google.com/scholar?q=Efficient+Estimation+of+Average+Treatment+Effects+Using+the+Estimated+Propensity+Score).
